# Improved Biodiversity Model

**Key improvements over the benchmark:**
1. All 14 TerraClimate variables (vs 2 in benchmark)
2. Multiple temporal aggregations per variable
3. LightGBM ensemble with 5-fold CV (vs Logistic Regression)
4. Threshold optimization
5. Spatial fallback features

**Prerequisite:** Run the TerraClimate_Extended.ipynb first to generate `TerraClimate_all_vars.tiff`.

In [ ]:
!pip install lightgbm xgboost rasterio -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import rasterio
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer
import lightgbm as lgb
import xgboost as xgb

print('Libraries loaded successfully')

## Step 1: Load data

In [ ]:
train = pd.read_csv('Training_Data.csv')
test  = pd.read_csv('Test.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Target distribution:')
print(train['Occurrence Status'].value_counts())
train.head()

## Step 2: Extract TerraClimate features from GeoTIFF

This reads `TerraClimate_all_vars.tiff` which contains all 14+ TerraClimate variables.
If that file is not available, falls back to `TerraClimate_output.tiff` (2 variables only).

In [ ]:
import os

# TerraClimate variable names (order matches bands in the extended GeoTIFF)
ALL_VARS = ['aet', 'def', 'pet', 'ppt', 'q', 'soil', 'srad', 'swe',
            'tmax', 'tmin', 'vap', 'ws', 'vpd', 'pdsi']

def extract_tiff_features(tiff_path, csv_path, var_names):
    """Extract raster values at coordinates given in CSV."""
    df = pd.read_csv(csv_path)

    with rasterio.open(tiff_path) as src:
        n_bands = src.count
        var_names = var_names[:n_bands]  # handle fewer bands gracefully

        # Build coordinate arrays
        lon = np.linspace(src.bounds.left, src.bounds.right, src.width)
        lat = np.linspace(src.bounds.top, src.bounds.bottom, src.height)

        band_das = []
        for b in range(1, n_bands + 1):
            data = src.read(b).astype(float)
            da = xr.DataArray(data, coords=[('lat', lat), ('lon', lon)])
            band_das.append(da)

    records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Extracting'):
        vals = {}
        for name, da in zip(var_names, band_das):
            try:
                v = da.sel(lat=row['Latitude'], lon=row['Longitude'],
                           method='nearest').values
                vals[name] = float(v)
            except Exception:
                vals[name] = np.nan
        records.append(vals)

    return pd.DataFrame(records)


# Choose GeoTIFF
if os.path.exists('TerraClimate_all_vars.tiff'):
    tiff_file = 'TerraClimate_all_vars.tiff'
    var_names  = ALL_VARS
    print('Using extended TerraClimate (14 vars)')
elif os.path.exists('TerraClimate_output.tiff'):
    tiff_file = 'TerraClimate_output.tiff'
    var_names  = ['srad', 'vap']
    print('WARNING: Using only srad+vap. Run TerraClimate_Extended.ipynb for better results.')
else:
    tiff_file = None
    var_names  = []
    print('No TerraClimate GeoTIFF found — using spatial-only features.')

if tiff_file:
    train_climate = extract_tiff_features(tiff_file, 'Training_Data.csv', var_names)
    test_climate  = extract_tiff_features(tiff_file, 'Test.csv', var_names)
else:
    train_climate = pd.DataFrame()
    test_climate  = pd.DataFrame()

## Step 3: Build spatial features

In [ ]:
from scipy.stats import gaussian_kde

train_coords = train[['Latitude', 'Longitude']].values
test_coords  = test[['Latitude', 'Longitude']].values
y_train      = train['Occurrence Status'].values

pos_coords = train_coords[y_train == 1]
neg_coords = train_coords[y_train == 0]

lat_m, lat_s = train_coords[:,0].mean(), train_coords[:,0].std()
lon_m, lon_s = train_coords[:,1].mean(), train_coords[:,1].std()

# KDE habitat suitability
n_pos, n_neg = len(pos_coords), len(neg_coords)
kde_pos = gaussian_kde(pos_coords.T, bw_method='silverman')
kde_neg = gaussian_kde(neg_coords.T, bw_method='silverman')

def kde_proba(coords):
    pv = kde_pos(coords.T) * n_pos
    nv = kde_neg(coords.T) * n_neg
    return pv / (pv + nv + 1e-12)


def build_spatial_features(coords):
    lat = coords[:, 0]
    lon = coords[:, 1]
    ln  = (lat - lat_m) / lat_s
    lo  = (lon - lon_m) / lon_s
    df = pd.DataFrame({
        'lat': lat, 'lon': lon,
        'ln': ln, 'lo': lo,
        'ln2': ln**2, 'lo2': lo**2,
        'ln3': ln**3, 'lo3': lo**3,
        'ln_lo': ln*lo, 'ln2_lo': ln**2*lo, 'ln_lo2': ln*lo**2,
        'dist_center': np.sqrt(ln**2 + lo**2),
        'sin_lat': np.sin(np.radians(lat)),
        'cos_lat': np.cos(np.radians(lat)),
        'sin_lon': np.sin(np.radians(lon)),
        'cos_lon': np.cos(np.radians(lon)),
        'kde_pos': kde_pos(coords.T),
        'kde_neg': kde_neg(coords.T),
        'kde_ratio': kde_proba(coords),
        'dist_pos_centroid': np.sqrt((lat - pos_coords[:,0].mean())**2 +
                                     (lon - pos_coords[:,1].mean())**2),
        'dist_neg_centroid': np.sqrt((lat - neg_coords[:,0].mean())**2 +
                                     (lon - neg_coords[:,1].mean())**2),
    })
    return df


train_spatial = build_spatial_features(train_coords)
test_spatial  = build_spatial_features(test_coords)

print('Spatial features:', train_spatial.shape[1])

## Step 4: Combine features

In [ ]:
def combine(spatial, climate):
    if len(climate) > 0:
        return pd.concat([spatial.reset_index(drop=True),
                          climate.reset_index(drop=True)], axis=1)
    return spatial.reset_index(drop=True)

X_train = combine(train_spatial, train_climate)
X_test  = combine(test_spatial,  test_climate)

# Impute missing values
imp = SimpleImputer(strategy='median')
X_train = pd.DataFrame(imp.fit_transform(X_train), columns=X_train.columns)
X_test  = pd.DataFrame(imp.transform(X_test),  columns=X_test.columns)

print(f'Feature matrix: train={X_train.shape}, test={X_test.shape}')
X_train.describe()

## Step 5: LightGBM ensemble with 5-fold CV

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lgb_params = dict(
    objective='binary', metric='binary_logloss',
    n_estimators=2000, learning_rate=0.03,
    num_leaves=127, max_depth=-1,
    min_child_samples=20, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.1,
    reg_lambda=0.1, verbose=-1, random_state=42,
)

xgb_params = dict(
    objective='binary:logistic', eval_metric='logloss',
    n_estimators=2000, learning_rate=0.03,
    max_depth=7, subsample=0.8,
    colsample_bytree=0.8, min_child_weight=5,
    reg_alpha=0.1, reg_lambda=1.0,
    early_stopping_rounds=50,
    random_state=42, verbosity=0,
)

oof_lgb = np.zeros(len(X_train))
oof_xgb = np.zeros(len(X_train))
test_lgb = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f'Fold {fold+1}/5...', end=' ')
    Xtr, Xval = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    ytr, yval = y_train[tr_idx], y_train[val_idx]

    # LightGBM
    m_lgb = lgb.LGBMClassifier(**lgb_params)
    m_lgb.fit(Xtr, ytr,
              eval_set=[(Xval, yval)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(-1)])
    oof_lgb[val_idx] = m_lgb.predict_proba(Xval)[:, 1]
    test_lgb += m_lgb.predict_proba(X_test)[:, 1] / 5

    # XGBoost
    m_xgb = xgb.XGBClassifier(**xgb_params)
    m_xgb.fit(Xtr, ytr, eval_set=[(Xval, yval)], verbose=False)
    oof_xgb[val_idx] = m_xgb.predict_proba(Xval)[:, 1]
    test_xgb += m_xgb.predict_proba(X_test)[:, 1] / 5

    lgb_acc = accuracy_score(yval, (oof_lgb[val_idx] > 0.5).astype(int))
    xgb_acc = accuracy_score(yval, (oof_xgb[val_idx] > 0.5).astype(int))
    print(f'LGB={lgb_acc:.4f}  XGB={xgb_acc:.4f}')

print()
print(f'LGB OOF: {accuracy_score(y_train, (oof_lgb > 0.5).astype(int)):.6f}')
print(f'XGB OOF: {accuracy_score(y_train, (oof_xgb > 0.5).astype(int)):.6f}')

## Step 6: Optimize blend weights and threshold

In [ ]:
best_acc, best_w, best_thresh = 0, 0.5, 0.5

for w in np.arange(0.0, 1.01, 0.05):
    blend = w * oof_lgb + (1 - w) * oof_xgb
    for thresh in np.arange(0.35, 0.65, 0.01):
        acc = accuracy_score(y_train, (blend > thresh).astype(int))
        if acc > best_acc:
            best_acc, best_w, best_thresh = acc, w, thresh

print(f'Best OOF accuracy: {best_acc:.6f}')
print(f'Best LGB weight: {best_w:.2f}, threshold: {best_thresh:.2f}')

## Step 7: Final predictions

In [ ]:
test_blend = best_w * test_lgb + (1 - best_w) * test_xgb
test_preds = (test_blend > best_thresh).astype(int)

print(f'Test positive: {test_preds.sum()} ({test_preds.mean():.3f})')

submission = pd.DataFrame({'ID': test['ID'], 'Target': test_preds})
submission.to_csv('submission_final.csv', index=False)
print('Saved: submission_final.csv')
submission.head(10)

## (Optional) Feature importance

In [ ]:
import matplotlib.pyplot as plt

# Train final LGB on all data for importance
final_lgb = lgb.LGBMClassifier(**lgb_params)
final_lgb.fit(X_train, y_train, callbacks=[lgb.log_evaluation(-1)])

imp_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': final_lgb.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 6))
plt.barh(imp_df['feature'][::-1], imp_df['importance'][::-1])
plt.xlabel('Feature Importance')
plt.title('Top 20 Features')
plt.tight_layout()
plt.show()

print(imp_df.to_string(index=False))